#### 4.2: Позиционные и именованные аргументы. Функции высших порядков. Лямбда-функции

final_price() фильтр: рассчитывать скидку только для цен в определённом диапазоне.

In [12]:
def final_price(*prices, discount=1, **kwargs):
    low = kwargs.get('price_low', min(prices))
    high = kwargs.get('price_high', max(prices))
    return [price - price * discount / 100 for price in prices if low <= price <= high]

print(final_price(100, 200, 300, 400, 500, discount=5))
print(final_price(100, 200, 300, 400, 500, discount=5, price_low=200))
print(final_price(100, 200, 300, 400, 500, discount=5, price_high=200))
print(final_price(100, 200, 300, 400, 500, discount=5, price_low=200, price_high=350))


[95.0, 190.0, 285.0, 380.0, 475.0]
[190.0, 285.0, 380.0, 475.0]
[95.0, 190.0]
[190.0, 285.0]


add_value() Функция добавляет значение x в список и возвращает его, значение параметра по умолчанию — список. 

In [16]:
def add_value(x, df = None):
    if df is None:
        df = []
    df += [x]
    return df

print(add_value(0))
print(add_value(0, [1, 2, 3]))
print(add_value(1))

[0]
[1, 2, 3, 0]
[1]


Присваивание параметру df значения по умолчанию `None` необходимо в данном случае, так как он является списком функции, если бы мы обозначили его списком по умолчанию `def add_value(x, df = []):`, после первого вызова он продолжил бы существовать и ссылаться на один объект памяти, тем самым при повтрном вызове использовались бы значения предыдущего вызова. Переменная объявленная в теле функции существует только в момент вызова оной, а список заданный по умолчанию сохраняется между вызовами. Данная особенность касается всех изменяемых типов данных.

1. Сортировка строк по длине

In [17]:
lines = ["abcd", "ab", "abc", "abcdef"]
print(sorted(lines, key=lambda x: len(x)))

['ab', 'abc', 'abcd', 'abcdef']


2. Сортировка сначала по длине, потом по алфавиту:

In [19]:
lines = ["abcd", "ab", "ba", "acde"]
print(sorted(lines, key=lambda x: (len(x), x)))

['ab', 'ba', 'abcd', 'acde']


3. Сортировка по убыванию длины, а при равенстве — по алфавиту:

In [20]:
lines = ["abcd", "ab", "ba", "acde"]
print(sorted(lines, key=lambda x: (-len(x), x)))

['abcd', 'acde', 'ab', 'ba']


`key` - именнованый аргумент, который принимает функцию для сравнения, в примерах выше это `lambda` функция с соответсвующими условиями, критерии идут в порядке написания, при неоднозначности первого, идет второй.

#### 4.3. Рекурсия. Декораторы. Генераторы

Декларативный

In [33]:
from timeit import timeit

def fib(n):
    if n in (0, 1):
        return 1
    return fib(n - 1) + fib(n - 2)

print(f"Среднее время вычисления: "
     f"{round(timeit('fib(35)', number=10, globals=globals()) / 10, 3)} с.")

Среднее время вычисления: 2.587 с.


Обычная рекурсия с большим количеством перерасчетов функции.

Императивный

In [34]:
from timeit import timeit

def fib(n):
    f_1, f = 1, 1
    for i in range(n - 1):
        f_1, f = f, f_1 + f
    return f

print(f"Среднее время вычисления: "
     f"{round(timeit('fib(35)', number=10, globals=globals()) / 10, 3)} с.")

Среднее время вычисления: 0.0 с.


Явно заданный последовательный алгоритм поиска числа, без избыточного перевызова.

Генератор

In [35]:
from timeit import timeit

def fib(n):
    n_1, n_2 = 1, 1
    for i in range(n):
        yield n_1
        n_1, n_2 = n_2, n_1 + n_2


print(f"Среднее время вычисления: "
     f"{round(timeit('fib(35)', number=10, globals=globals()) / 10, 3)} с.")

Среднее время вычисления: 0.0 с.


yield - оператор, который приостанавливает выполнение, сохраняет значение перменной и передает его при вызове.

Кэширование через словарь

In [ ]:
from timeit import timeit
from sys import setrecursionlimit

def fib(n):
    if n not in cache:
        cache[n] = fib(n - 1) + fib(n - 2)
    return cache[n]

setrecursionlimit(2000)
cache = {0: 1, 1: 1}
print(f"Среднее время вычисления: "
     f"{round(timeit('fib(1000)', number=10, globals=globals()) / 10, 6)} с.")

Среднее время вычисления: 6.3e-05 с.


Мы сами вводим и сохраняем в переменную промежуточные значения, в данном случае в словарь.

Кеширование с помощью lru_cache

In [36]:
from timeit import timeit
from functools import lru_cache

@lru_cache(maxsize=1000)
def fib(n):
    if n in (0, 1):
        return 1
    return fib(n - 1) + fib(n - 2)

print(f'Среднее время вычисления: '
    f'{round(timeit('fib(35)', number=10, globals=globals()) / 10, 6)} c.')
print(fib.cache_info())
print(f'Среднее время вычисления: '
    f'{round(timeit('fib(35)', number=10, globals=globals()) / 10, 6)} c.')
print(fib.cache_info())

Среднее время вычисления: 3e-06 c.
CacheInfo(hits=42, misses=36, maxsize=1000, currsize=36)
Среднее время вычисления: 0.0 c.
CacheInfo(hits=52, misses=36, maxsize=1000, currsize=36)


lru_cache - встроенный в модуль functools декоратор, который автоматически запоминает результаты предыдущих вызовов функции и позволяет их использовать.

Декораторы

In [ ]:
# декратор принимаем функцию как аргумент
def count(f):
    total = 0
# объявляем функцию, которая расширяет функционал f
    def decorated(*args, **kwargs):
        # Переменная total объявлена нелокальной для доступа из внутренней функции
        nonlocal total
        total += 1
        # Возвращаем значение исходной функции и дополнительно total
        return f(*args, **kwargs), total
    # Возвращаем новую функцию как объект 
    return decorated

@count
def hello(name):
    return f"Привет, {name}!"


print(hello("Пользователь_1"))
print(hello("Пользователь_2"))

('Привет, Пользователь_1!', 1)
('Привет, Пользователь_2!', 2)


Тут сам практиковался

In [37]:
def decorator(func):
    def before(*args, **kwargs):
        print("выводим до")
        return func(*args, **kwargs)

    return before


@decorator
def sum_two(a, b):
    return a + b


print(sum_two(2, 3))

выводим до
5
